# Notebook 5: Renaming and combining tables
*Kaggle Pandas lesson: "Renaming and Combining"*

**Module question: what makes a stock risky?** Real analysis rarely lives in one table. Today we add
three more files and combine them with our snapshot:

* `fundamentals_2024.csv`: last year's income statement, with **raw Compustat names** (`sale`, `oiadp`, `ni`...).
  Joining it lets us compute growth and the *realized operating leverage* of each firm.
* `small_caps_2025.csv`: the firms between $300M and $1B that our snapshot left out, with two columns
  named slightly differently. Stacking them on reveals the size effect.
* `gics_industries.csv`: the names of the 6-digit industry codes, for a finer look than sectors.

## Learning goals
* rename columns and index labels (`rename`, `rename_axis`);
* stack tables vertically with `pd.concat`;
* combine tables side by side on a shared key with `join` (and know that `merge` does the same on columns).

## Setup

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/assacohen1/fin6040-pandas-data/main/"
firms = pd.read_csv(DATA_URL + "companies_2025.csv")
f24   = pd.read_csv(DATA_URL + "fundamentals_2024.csv")
small = pd.read_csv(DATA_URL + "small_caps_2025.csv")
gics  = pd.read_csv(DATA_URL + "gics_industries.csv")

pd.set_option("display.max_columns", 45)
pd.set_option("display.max_rows", 30)

## In class

### 1. Look before you combine

In [ ]:
f24.head()

In [ ]:
f24.columns

In [ ]:
small.columns

Compustat's raw names (`sale`, `xsga`, `oiadp`, `ni`, `at`) are cryptic, and the small-cap file calls
market cap `mktcap` and volatility `vol`. Tables must agree on names before they are combined, so we rename first.

### 2. `rename`: columns and index labels
`rename(columns={old: new, ...})` takes a dictionary and returns a new DataFrame (assign it back to keep it).

In [ ]:
f24.rename(columns={"tic": "ticker", "sale": "sales"}).head()

It can rename index labels too, though that is rarely useful; `set_index` is the usual way to get meaningful row labels.

In [ ]:
firms.rename(index={0: "largest", 1: "second"}).head(3)

`rename_axis` names the index itself:

In [ ]:
firms.rename_axis("firm_no", axis="rows").head(3)

### 3. `pd.concat`: stacking rows
`concat` takes a list of DataFrames and stacks them. Columns are matched **by name**; a column that exists in
only one table gets NaN in the other's rows (which is exactly what would happen with `mktcap` versus
`market_cap`). The original index labels are kept unless you ask for a fresh 0, 1, 2, ... with `ignore_index=True`.

In [ ]:
pd.concat([firms.head(2), firms.tail(2)])

In [ ]:
pd.concat([firms.head(2), firms.tail(2)], ignore_index=True)

### 4. `join`: combining columns on a shared key
`join` lines two tables up **by index label**, so we first `set_index` both on the key (`ticker`).
Rows with no match on the right get NaN (a *left* join). When both tables have a column with the same
name, `lsuffix` / `rsuffix` tell them apart. A small demonstration with sales and EBIT in both years:

In [ ]:
left = firms.set_index("ticker")[["sales", "ebit"]]
right = f24.rename(columns={"tic": "ticker", "sale": "sales", "oiadp": "ebit"}).set_index("ticker")[["sales", "ebit"]]
demo = left.join(right, lsuffix="_2025", rsuffix="_2024")
demo.head()

Once the index is the ticker, `loc` looks firms up by name:

In [ ]:
demo.loc["AAPL"]

`pd.merge(firms, f24, left_on="ticker", right_on="tic", how="left")` does the same job on ordinary
columns instead of the index; we stick with `join` here, as in the Kaggle lesson.

## Exercises

### Exercise 1: Readable names

Rename the columns of `f24` so that `tic` becomes `ticker`, `sale` becomes `sales_2024`, `cogs` becomes
`cogs_2024`, `xsga` becomes `sga_2024`, `oiadp` becomes `ebit_2024`, `ni` becomes `net_income_2024` and
`at` becomes `total_assets_2024`. Save the result back into `f24`.

<details><summary>Hint</summary>

A dictionary `{old: new}` with seven entries.

</details>

In [ ]:
f24 = f24.rename(columns={____})

f24.head()

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert "ticker" in f24.columns and "sale" not in f24.columns
assert {"sales_2024", "ebit_2024", "net_income_2024", "total_assets_2024"} <= set(f24.columns)
print("Looks right!")

### Exercise 2: Name the index

Create `firms_idx`: `firms` with `ticker` as the index, and the index named `symbol`.

<details><summary>Hint</summary>

`set_index(...)` then `rename_axis(..., axis="rows")`.

</details>

In [ ]:
firms_idx = ____

firms_idx.head(3)

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert firms_idx.index.name == "symbol"
assert "AAPL" in firms_idx.index
print("Looks right!")

### Exercise 3: Join the two years

Create `merged` by joining `firms` (index `ticker`) with the renamed `f24` (index `ticker`). How many firms have no 2024 data (`sales_2024` missing)? Assign that count to `n_new`.

<details><summary>Hint</summary>

`firms.set_index("ticker").join(f24.set_index("ticker"))`; then `isnull().sum()` on the new column.

</details>

In [ ]:
merged = ____
n_new = ____

print(n_new, "firms have no fiscal-2024 data")
merged.head()

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert merged.shape[0] == len(firms)
assert "sales_2024" in merged.columns
assert n_new == 4
print("Looks right!")

### Exercise 4: Growth and realized operating leverage

In `merged`, create three columns: `sales_growth` = `sales / sales_2024 - 1`, `ebit_growth` =
`ebit / ebit_2024 - 1`, and `dol` = `ebit_growth / sales_growth` (the *degree of operating leverage*:
how many percent EBIT moved for each percent of sales). The ratio is meaningless when either EBIT is
negative or sales barely moved, so build one True/False Series `bad` that is True when `ebit_2024 <= 0`,
or `ebit <= 0`, or `sales_2024 <= 0`, or the absolute value of `sales_growth` is below 0.05, and set
`dol` to NaN for those rows (`loc`, as in Notebook 4). Assign the median `sales_growth` to `median_growth`.

<details><summary>Hint</summary>

Four conditions joined with `|`, each in parentheses; `.abs()` gives absolute values.

</details>

In [ ]:
merged["sales_growth"] = ____
merged["ebit_growth"] = ____
merged["dol"] = ____
bad = ____
merged.loc[bad, "dol"] = float("nan")
median_growth = ____

print(f"median sales growth {median_growth:.1%}; {merged.dol.notnull().sum()} firms with a meaningful DOL")
merged.loc[~bad, ["sales_growth", "ebit_growth", "dol"]].describe()   # meaningful rows only

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert round(median_growth, 4) == 0.0721
assert merged.dol.notnull().sum() == 1071
assert merged.loc[merged.ebit_2024 <= 0, "dol"].isnull().all()
print("Looks right!")

### Exercise 5: Does operating leverage show up in stock risk?

Create `abs_dol = merged.dol.abs()`. Assign `high_dol_vol`, the median `vol_2025` of firms with
`abs_dol` above 2, and `low_dol_vol`, the median for firms with `abs_dol` of 2 or less.
Which group is riskier?

<details><summary>Hint</summary>

`merged.loc[abs_dol > 2, "vol_2025"].median()`; rows where `abs_dol` is NaN fail both conditions and are left out, which is what we want.

</details>

In [ ]:
abs_dol = ____
high_dol_vol = ____
low_dol_vol = ____

print(f"|DOL| > 2: {high_dol_vol:.3f}    |DOL| <= 2: {low_dol_vol:.3f}")

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert round(high_dol_vol, 4) == 0.4114
assert high_dol_vol > low_dol_vol
print("Looks right!")

### Exercise 6: Add the small caps: the size effect

Rename the two odd columns of `small` so they match `firms` (`mktcap` to `market_cap`, `vol` to
`vol_2025`). Add a column `size_group` equal to `"large"` in `firms` and `"small"` in `small`. Create
`all_firms` by stacking the two tables with a fresh index, then `size_vol`, the median `vol_2025` by `size_group`.

<details><summary>Hint</summary>

`rename(columns={...})`, `pd.concat([firms, small], ignore_index=True)`, then a `groupby` from Notebook 3.

</details>

In [ ]:
small = ____
firms["size_group"] = "large"
small["size_group"] = ____
all_firms = ____
size_vol = ____

print(len(all_firms), "firms in total")
size_vol

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert all_firms.shape[0] == len(firms) + len(small)
assert list(all_firms.columns) == list(firms.columns)     # a forgotten rename adds extra columns
assert all_firms.vol_2025.isnull().sum() == 0
assert size_vol["small"] > size_vol["large"]
assert round(size_vol["small"], 4) == 0.539
print("Looks right!")

### Exercise 7: Industry names

Join `gics` onto `firms` using `industry_code` as the index on both sides, then `reset_index()`; call the
result `with_ind`. Create `ind_vol`, the median `vol_2025` by `industry_name`, most volatile first, and
show the top 10 and the bottom 10 (`head(10)`, `tail(10)`).

<details><summary>Hint</summary>

`firms.set_index("industry_code").join(gics.set_index("industry_code")).reset_index()`.

</details>

In [ ]:
with_ind = ____
ind_vol = ____

display(ind_vol.head(10))
ind_vol.tail(10)

In [ ]:
# Self-check: run this cell after your answer. No error means you are right.
assert with_ind.industry_name.isnull().sum() == 0
assert len(with_ind) == len(firms)
assert ind_vol.index[0] == 'Biotechnology'
assert ind_vol.index[-1] == 'Multi-Utilities'
print("Looks right!")

*Optional:* the industry ranking as a chart (self-contained: it rebuilds what it needs).

In [ ]:
ind = firms.set_index("industry_code").join(gics.set_index("industry_code")).reset_index()
ind.groupby("industry_name").vol_2025.median().sort_values().tail(15).plot(kind="barh", title="Most volatile industries, 2025 (median)");

## Finance insight (and the module in one paragraph)

Two more drivers, then the full picture.

**Size.** Firms worth $300M to $1B have a median volatility of 0.54 against
0.39 for firms above $1B. The small-cap "premium" that asset pricing talks about
starts with plain higher risk.

**Operating leverage.** Firms whose EBIT moved more than twice as much as their sales (|DOL| above 2)
have a median volatility of 0.41 against 0.34 for the rest.
Fixed costs turn a small revenue surprise into a large earnings surprise, and the stock price follows.

**Industry.** The most volatile industry is Biotechnology (median 0.66), the
calmest is Multi-Utilities (0.20): the sector story from Notebook 3, sharpened.

Putting the five notebooks together: a stock's total risk is built from **business risk** (sector and
industry), **cost structure** (operating leverage), **capital structure** (financial leverage, which firms
choose in response to their business risk), **size**, **age** and **profitability**. Beta captures only the
part of that risk that moves with the market.